# ENGG2112: CNN-BiLSTM recurrent model

Train a bidirectional LSTM classifier over spatial features from a frozen ImageNet ResNet-18 encoder. This is a **single-model notebook**; it does not train the Random Forest or fine-tuned ResNet-18 baseline. It uses the same prepared handoff and fixed split, reports three-seed validation results, and keeps the test split locked pending review and freeze.

**Execution provenance (1 November 2026):** saved outputs below are retained from the earlier model runs. The path/cache maintenance changes have been checked separately; the models have not been retrained for this maintenance change. These are historical validation results, not final test results. Run all cells in order with your authorised data to generate a new run. See [current status and remaining evaluation work](../docs/PROJECT_STATUS.md).


## Setup and configuration

Use the prepared APTOS handoff and existing group-aware split. This notebook does not download data or create a new split. Set `DATA_ROOT` to the prepared handoff root if it is not at the default path. PyTorch, TorchVision, and scikit-learn must be installed in the selected notebook kernel. The first run may download official ImageNet ResNet-18 weights; do not disable certificate checks.

Open this notebook from the complete repository. Paths are resolved from the repository root, including when the kernel starts inside `jupyter notebooks/`. Edit `PROJECT_ROOT` and `DATA_ROOT` for external data. RF and CNN-BiLSTM use `src/model_support.py` from this checkout.


In [1]:
from pathlib import Path
import hashlib
import json
import random

import numpy as np
import pandas as pd
from PIL import Image

try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
    from torchvision import models, transforms
    from torchvision.models import ResNet18_Weights
    from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                                 confusion_matrix, cohen_kappa_score, f1_score)
except ImportError as exc:
    raise ImportError(
        'Install PyTorch, TorchVision, NumPy, pandas, Pillow, and scikit-learn '
        'in the selected notebook kernel, then restart the kernel.'
    ) from exc

SEED = 2112
REPEAT_SEEDS = [2112, 2113, 2114]
def find_project_root(start=None):
    """Find the repository or extracted handoff from the kernel's working folder."""
    start = Path.cwd() if start is None else Path(start)
    start = start.resolve()
    for candidate in (start, *start.parents):
        repository = (candidate / 'requirements.txt').is_file() and (candidate / 'src').is_dir()
        handoff = ((candidate / 'data/train.csv').is_file()
                   and (candidate / 'artifacts/splits').is_dir())
        if repository or handoff:
            return candidate
    raise FileNotFoundError(
        'Cannot locate the project. Set PROJECT_ROOT to the absolute repository '
        'or prepared handoff folder in this configuration cell.'
    )

PROJECT_ROOT = find_project_root()  # Or Path(r'/absolute/path/to/project').resolve()

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_ROOT = PROJECT_ROOT / 'artifacts/notebook_runs/notebook_run_001'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'aptos_cnn_bilstm'
FEATURE_CACHE = PROJECT_ROOT / 'outputs' / 'aptos_shared_features' / 'resnet18_grid_features.npz'
BATCH_SIZE = 32
RNN_MAX_EPOCHS = 20
RNN_PATIENCE = 5
RNN_LEARNING_RATE = 1e-3
RNN_WEIGHT_DECAY = 1e-4
CONFIRM_REVIEWED_FROZEN_SPLIT = False
TEST_EVALUATION_COMPLETED = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('mps' if torch.backends.mps.is_available()
                      else 'cuda' if torch.cuda.is_available() else 'cpu')

SPLIT_PATH = DATA_ROOT / 'artifacts/splits/split_manifest.csv'
PREPARED_DIR = DATA_ROOT / 'data/processed_224_v1'
PREPARED_MANIFEST_PATH = PREPARED_DIR / 'processed_manifest.csv'
PREPROCESSING_CONFIG_PATH = PREPARED_DIR / 'preprocessing_config.json'
missing = [str(path) for path in (SPLIT_PATH, PREPARED_MANIFEST_PATH, PREPROCESSING_CONFIG_PATH)
           if not path.is_file()]
if missing:
    raise FileNotFoundError('Prepared APTOS handoff is incomplete. Set DATA_ROOT to the handoff root. Missing: '
                            + ', '.join(missing))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_CACHE.parent.mkdir(parents=True, exist_ok=True)
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

MANIFEST_SHA256 = sha256_file(SPLIT_PATH)
print('Device:', DEVICE, '| Torch:', torch.__version__, '| TorchVision:', __import__('torchvision').__version__)
print('Handoff:', DATA_ROOT, '| Split manifest SHA-256:', MANIFEST_SHA256)

Device: mps | Torch: 2.14.1 | TorchVision: 0.29.1
Handoff: /Users/quangduyvu1723/Documents/GitHub/ENGG2112-final-project/artifacts/notebook_runs/notebook_run_001 | Split manifest SHA-256: 81283b51d8274a7d4ee5cd873cfc6479fd9efd178a75bacd45e297dd41d57a54


## Shared frozen image features

Both alternative classifiers consume deterministic features from the same ImageNet-pretrained, frozen ResNet-18 encoder. For each 224 × 224 image the encoder creates 49 spatial cells with 512 channels. The Random Forest receives the mean across the 49 cells (512 features). The CNN-BiLSTM receives all 49 feature vectors in row-major order. No labels are used to fit the encoder. The cache is local, ignored by Git, and checked against the split and prepared manifests, preprocessing configuration, actual train/validation image bytes, encoder and transform settings, IDs, labels, and feature shape. Legacy or invalid caches are rebuilt automatically. Test image bytes are not read by this check.


In [2]:
split = pd.read_csv(SPLIT_PATH, dtype={'id_code': str, 'group_id': str, 'pixel_sha256': str})
prepared = pd.read_csv(PREPARED_MANIFEST_PATH, dtype={'id_code': str})
split_columns = {'id_code', 'diagnosis', 'split', 'group_id', 'pixel_sha256'}
prepared_columns = {'id_code', 'diagnosis', 'split', 'relative_path'}
if not split_columns <= set(split.columns) or not prepared_columns <= set(prepared.columns):
    raise ValueError('Handoff manifests do not satisfy the expected columns.')
for label, frame in [('split', split), ('prepared', prepared)]:
    if frame.id_code.isna().any() or frame.id_code.duplicated().any():
        raise ValueError(f'{label} manifest contains missing or duplicate IDs.')
    if not pd.to_numeric(frame.diagnosis, errors='coerce').isin(range(5)).all():
        raise ValueError(f'{label} manifest contains an invalid grade.')
    if not frame.split.isin(['train', 'validation', 'test']).all():
        raise ValueError(f'{label} manifest contains an invalid split.')
if set(split.id_code) != set(prepared.id_code):
    raise ValueError('Split and prepared manifests have different image IDs.')
joined = split.merge(prepared, on='id_code', validate='one_to_one', suffixes=('', '_prepared'))
if not joined.diagnosis.eq(joined.diagnosis_prepared).all() or not joined.split.eq(joined.split_prepared).all():
    raise ValueError('Split and prepared manifests disagree on labels or assignments.')
for column in ('group_id', 'pixel_sha256'):
    if split.groupby(column).split.nunique().max() != 1:
        raise ValueError(f'{column} crosses split boundaries.')
if not joined.relative_path.eq(joined.id_code + '.png').all():
    raise ValueError('Prepared image paths must be ID.png.')
preprocessing = json.loads(PREPROCESSING_CONFIG_PATH.read_text(encoding='utf-8'))
if preprocessing.get('size') != 224 or preprocessing.get('mode') != 'RGB':
    raise ValueError('Expected prepared 224 × 224 RGB images.')
joined['diagnosis'] = joined.diagnosis.astype(int)
joined['image_path'] = [str(PREPARED_DIR / name) for name in joined.relative_path]
missing_images = [path for path in joined.image_path if not Path(path).is_file()]
if missing_images:
    raise FileNotFoundError(f'{len(missing_images)} prepared images are missing.')
joined = joined.sort_values('id_code').reset_index(drop=True)
train_frame = joined.loc[joined.split.eq('train')].reset_index(drop=True)
validation_frame = joined.loc[joined.split.eq('validation')].reset_index(drop=True)
test_frame = joined.loc[joined.split.eq('test')].reset_index(drop=True)
if min(len(train_frame), len(validation_frame), len(test_frame)) == 0:
    raise ValueError('Train, validation, and test splits must all be non-empty.')
print('Rows:', len(joined), '| train:', len(train_frame), '| validation:', len(validation_frame),
      '| test locked:', len(test_frame))
display(pd.crosstab(joined.split, joined.diagnosis).reindex(columns=range(5), fill_value=0))

ENCODER_ID = ResNet18_Weights.DEFAULT.name
feature_frame = joined.loc[joined.split.isin(['train', 'validation'])].reset_index(drop=True)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

class ImageDataset(torch.utils.data.Dataset):
    def __init__(self, frame):
        self.frame = frame
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row.image_path) as image:
            image = image.convert('RGB')
            if image.size != (224, 224):
                raise ValueError(f'Unexpected image size for {row.id_code}: {image.size}')
            return self.transform(image)

_encoder = None
def extract_feature_sequences(frame):
    global _encoder
    if _encoder is None:
        encoder_model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
        _encoder = nn.Sequential(*list(encoder_model.children())[:-2]).to(DEVICE).eval()
        for parameter in _encoder.parameters():
            parameter.requires_grad_(False)
    image_loader = DataLoader(ImageDataset(frame), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    batches = []
    with torch.inference_mode():
        for batch_number, images in enumerate(image_loader, start=1):
            fmap = _encoder(images.to(DEVICE))
            batches.append(fmap.flatten(2).transpose(1, 2).cpu().numpy().astype(np.float32))
            if batch_number % 20 == 0 or batch_number == len(image_loader):
                print(f'Encoded {min(batch_number * BATCH_SIZE, len(frame))}/{len(frame)} permitted images')
    return np.concatenate(batches, axis=0)

from src.model_support import feature_cache_metadata, load_feature_cache

# Hash actual train/validation image bytes; the test images remain unopened.
cache_metadata = feature_cache_metadata(
    feature_frame, SPLIT_PATH, PREPARED_MANIFEST_PATH, PREPROCESSING_CONFIG_PATH,
    encoder={'architecture': 'resnet18', 'weights': ENCODER_ID,
             'torch': torch.__version__, 'torchvision': __import__('torchvision').__version__},
    transform={'mode': 'RGB', 'size': [224, 224], 'tensor_scale': 'uint8 / 255',
               'mean': list(IMAGENET_MEAN), 'std': list(IMAGENET_STD), 'augmentation': 'none'},
)
feature_ids = feature_frame.id_code.to_numpy(dtype=str)
feature_labels = feature_frame.diagnosis.to_numpy(dtype=np.int64)
cached_arrays = load_feature_cache(FEATURE_CACHE, cache_metadata, feature_ids, feature_labels)
if cached_arrays is not None:
    feature_sequences, feature_labels, feature_ids = cached_arrays
    print('Reused validated train/validation feature cache:', FEATURE_CACHE)
else:
    feature_sequences = extract_feature_sequences(feature_frame)
    if (feature_sequences.shape != (len(feature_frame), 49, 512)
            or feature_sequences.dtype != np.float32 or not np.isfinite(feature_sequences).all()):
        raise ValueError('Encoder produced invalid feature arrays; cache was not written.')
    np.savez_compressed(FEATURE_CACHE, features=feature_sequences, labels=feature_labels,
                        id_code=feature_ids, metadata=json.dumps(cache_metadata))
    print('Rebuilt train/validation feature cache for the current inputs:', FEATURE_CACHE)

split_indices = {name: np.flatnonzero(feature_frame.split.to_numpy() == name)
                  for name in ('train', 'validation')}
X_sequence = {name: feature_sequences[idx] for name, idx in split_indices.items()}
y = {name: feature_labels[idx] for name, idx in split_indices.items()}
X_pooled = {name: values.mean(axis=1) for name, values in X_sequence.items()}
print('Test images have not been opened or encoded.')
print('Pooled RF feature shape:', X_pooled['train'].shape,
      '| CNN-BiLSTM sequence shape:', X_sequence['train'].shape)

Rows: 3600 | train: 2521 | validation: 538 | test locked: 541


diagnosis,0,1,2,3,4
split,,,,,
test,273,53,144,27,44
train,1258,253,684,127,199
validation,270,54,148,26,40


Reused validated train/validation feature cache: /Users/quangduyvu1723/Documents/GitHub/ENGG2112-final-project/outputs/aptos_shared_features/resnet18_grid_features.npz
Test images have not been opened or encoded.
Pooled RF feature shape: (2521, 512) | CNN-BiLSTM sequence shape: (2521, 49, 512)


## CNN-BiLSTM method

A recurrent network is not naturally suited to unordered pixels. Here, a bidirectional LSTM reads the 49 spatial feature vectors from a frozen CNN feature map as an ordered row-major sequence. Mean and max sequence pooling feed a five-grade classifier. Only the BiLSTM head is trained; the shared ResNet encoder remains frozen. Three predefined seeds assess variation. For each seed, early stopping and checkpoint selection use validation macro-F1. Seed 2112 is the predeclared primary model for any eventual test evaluation.


In [3]:
class SpatialBiLSTM(nn.Module):
    def __init__(self, input_size=512, hidden_size=128, num_classes=5, dropout=0.25):
        super().__init__()
        self.recurrent = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                                 batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * 4, num_classes)

    def forward(self, sequence):
        outputs, _ = self.recurrent(sequence)
        pooled = torch.cat((outputs.mean(dim=1), outputs.max(dim=1).values), dim=1)
        return self.classifier(self.dropout(pooled))

RNN_DEVICE = DEVICE
try:
    _probe = SpatialBiLSTM().to(RNN_DEVICE)
    _probe_input = torch.zeros((2, 49, 512), dtype=torch.float32, device=RNN_DEVICE, requires_grad=True)
    _probe(_probe_input).sum().backward()
    del _probe
except (NotImplementedError, RuntimeError) as exc:
    if DEVICE.type != 'mps':
        raise
    RNN_DEVICE = torch.device('cpu')
    print(f'LSTM unavailable on MPS ({type(exc).__name__}); using CPU for the recurrent head.')
print('CNN feature extraction device:', DEVICE, '| Recurrent training device:', RNN_DEVICE)

def make_sequence_loaders(seed):
    generator = torch.Generator().manual_seed(seed)
    train_data = TensorDataset(torch.from_numpy(X_sequence['train']), torch.from_numpy(y['train']))
    val_data = TensorDataset(torch.from_numpy(X_sequence['validation']), torch.from_numpy(y['validation']))
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=0, generator=generator)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    return train_loader, val_loader

def predict_sequence_model(model, loader):
    model.eval()
    true_values, predictions = [], []
    with torch.inference_mode():
        for features, labels in loader:
            logits = model(features.to(RNN_DEVICE))
            true_values.extend(labels.numpy().tolist())
            predictions.extend(logits.argmax(dim=1).cpu().numpy().tolist())
    return np.asarray(true_values), np.asarray(predictions)

class_counts = np.bincount(y['train'], minlength=5)
if (class_counts == 0).any():
    raise ValueError('All five grades must occur in the training split.')
class_weights = len(y['train']) / (5.0 * class_counts)
rnn_results = []
rnn_primary_validation = None
primary_checkpoint_path = OUTPUT_DIR / 'cnn_bilstm_seed_2112_best.pt'

def calculate_metrics(true, pred):
    metrics = {
        'accuracy': accuracy_score(true, pred),
        'balanced_accuracy': balanced_accuracy_score(true, pred),
        'macro_f1': f1_score(true, pred, average='macro'),
        'quadratic_weighted_kappa': cohen_kappa_score(true, pred, weights='quadratic'),
        'per_class': classification_report(true, pred, output_dict=True),
        'confusion_matrix': confusion_matrix(true, pred).tolist(),
    }
    return metrics

for seed in REPEAT_SEEDS:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    train_loader, val_loader = make_sequence_loaders(seed)
    recurrent_model = SpatialBiLSTM().to(RNN_DEVICE)
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(class_weights, dtype=torch.float32, device=RNN_DEVICE))
    optimizer = torch.optim.AdamW(recurrent_model.parameters(), lr=RNN_LEARNING_RATE,
                                  weight_decay=RNN_WEIGHT_DECAY)
    best_macro_f1 = -1.0
    best_epoch = 0
    no_improvement = 0
    run_checkpoint = OUTPUT_DIR / f'cnn_bilstm_seed_{seed}_best.pt'
    for epoch in range(1, RNN_MAX_EPOCHS + 1):
        recurrent_model.train()
        for features, labels in train_loader:
            features, labels = features.to(RNN_DEVICE), labels.to(RNN_DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(recurrent_model(features), labels)
            loss.backward()
            optimizer.step()
        val_true, val_pred = predict_sequence_model(recurrent_model, val_loader)
        val_metrics = calculate_metrics(val_true, val_pred)
        print(f'BiLSTM seed {seed}, epoch {epoch}: val_macro_f1={val_metrics["macro_f1"]:.4f}, '
              f'val_QWK={val_metrics["quadratic_weighted_kappa"]:.4f}')
        if val_metrics['macro_f1'] > best_macro_f1:
            best_macro_f1 = val_metrics['macro_f1']
            best_epoch = epoch
            no_improvement = 0
            torch.save({'model_state_dict': recurrent_model.state_dict(), 'seed': seed,
                        'epoch': epoch, 'validation_metrics': val_metrics,
                        'manifest_sha256': MANIFEST_SHA256}, run_checkpoint)
        else:
            no_improvement += 1
            if no_improvement >= RNN_PATIENCE:
                print(f'Early stopping seed {seed} at epoch {epoch}; best epoch {best_epoch}.')
                break
    saved = torch.load(run_checkpoint, map_location=RNN_DEVICE, weights_only=True)
    recurrent_model.load_state_dict(saved['model_state_dict'])
    val_true, val_pred = predict_sequence_model(recurrent_model, val_loader)
    val_metrics = calculate_metrics(val_true, val_pred)
    rnn_results.append({'method': 'CNN-BiLSTM', 'seed': seed, 'best_epoch': best_epoch,
                        **{key: val_metrics[key] for key in ('accuracy', 'balanced_accuracy',
                                                            'macro_f1', 'quadratic_weighted_kappa')},
                        'weighted_f1': val_metrics['per_class']['weighted avg']['f1-score']})
    if seed == REPEAT_SEEDS[0]:
        rnn_primary_validation = val_metrics
        torch.save(saved, primary_checkpoint_path)

CLASS_NAMES = [f'Grade {i}' for i in range(5)]
rnn_results_frame = pd.DataFrame(rnn_results)
display(rnn_results_frame)
display(pd.DataFrame(rnn_primary_validation['per_class']).T)
display(pd.DataFrame(rnn_primary_validation['confusion_matrix'], index=CLASS_NAMES, columns=CLASS_NAMES))

metric_names = ['accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1', 'quadratic_weighted_kappa']
rnn_results_frame = pd.DataFrame(rnn_results)
rnn_summary = {
    metric: {'mean': float(rnn_results_frame[metric].mean()),
             'std': float(rnn_results_frame[metric].std(ddof=1))}
    for metric in metric_names
}
rnn_payload = {
    'model': 'CNN-BiLSTM on spatial frozen ResNet-18 features',
    'split_manifest_sha256': MANIFEST_SHA256,
    'encoder': ENCODER_ID,
    'feature_cache_metadata': cache_metadata,
    'device': str(RNN_DEVICE),
    'repeat_seeds': REPEAT_SEEDS,
    'primary_seed': REPEAT_SEEDS[0],
    'settings': {'hidden_size': 128, 'bidirectional': True, 'batch_size': BATCH_SIZE,
                 'max_epochs': RNN_MAX_EPOCHS, 'patience': RNN_PATIENCE,
                 'learning_rate': RNN_LEARNING_RATE, 'weight_decay': RNN_WEIGHT_DECAY,
                 'selection': 'validation macro-F1'},
    'validation_runs': rnn_results,
    'validation_summary': rnn_summary,
    'primary_validation': rnn_primary_validation,
    'test_evaluation': 'not run; review and freeze the split first',
}
RNN_RESULTS_PATH = OUTPUT_DIR / 'validation_metrics.json'
RNN_RESULTS_PATH.write_text(json.dumps(rnn_payload, indent=2), encoding='utf-8')
print('CNN-BiLSTM validation results saved to:', RNN_RESULTS_PATH)


CNN feature extraction device: mps | Recurrent training device: mps


/Users/quangduyvu1723/Documents/GitHub/ENGG2112-final-project/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/quangduyvu1723/Documents/GitHub/ENGG2112-final-project/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/quangduyvu1723/Documents/GitHub/ENGG2112-final-project/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples.

BiLSTM seed 2112, epoch 1: val_macro_f1=0.3404, val_QWK=0.7730
BiLSTM seed 2112, epoch 2: val_macro_f1=0.4909, val_QWK=0.7923
BiLSTM seed 2112, epoch 3: val_macro_f1=0.5065, val_QWK=0.7735
BiLSTM seed 2112, epoch 4: val_macro_f1=0.5348, val_QWK=0.8047
BiLSTM seed 2112, epoch 5: val_macro_f1=0.5741, val_QWK=0.8207
BiLSTM seed 2112, epoch 6: val_macro_f1=0.5504, val_QWK=0.8030
BiLSTM seed 2112, epoch 7: val_macro_f1=0.5870, val_QWK=0.8310
BiLSTM seed 2112, epoch 8: val_macro_f1=0.5908, val_QWK=0.8157
BiLSTM seed 2112, epoch 9: val_macro_f1=0.5934, val_QWK=0.8529
BiLSTM seed 2112, epoch 10: val_macro_f1=0.6189, val_QWK=0.8208
BiLSTM seed 2112, epoch 11: val_macro_f1=0.5920, val_QWK=0.8336
BiLSTM seed 2112, epoch 12: val_macro_f1=0.6015, val_QWK=0.8440
BiLSTM seed 2112, epoch 13: val_macro_f1=0.5846, val_QWK=0.8273
BiLSTM seed 2112, epoch 14: val_macro_f1=0.6049, val_QWK=0.8295
BiLSTM seed 2112, epoch 15: val_macro_f1=0.5778, val_QWK=0.8254
Early stopping seed 2112 at epoch 15; best epoch 

/Users/quangduyvu1723/Documents/GitHub/ENGG2112-final-project/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/quangduyvu1723/Documents/GitHub/ENGG2112-final-project/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/quangduyvu1723/Documents/GitHub/ENGG2112-final-project/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples.

BiLSTM seed 2113, epoch 4: val_macro_f1=0.4312, val_QWK=0.7810
BiLSTM seed 2113, epoch 5: val_macro_f1=0.5798, val_QWK=0.8135
BiLSTM seed 2113, epoch 6: val_macro_f1=0.5523, val_QWK=0.8193
BiLSTM seed 2113, epoch 7: val_macro_f1=0.6091, val_QWK=0.8355
BiLSTM seed 2113, epoch 8: val_macro_f1=0.5508, val_QWK=0.8219
BiLSTM seed 2113, epoch 9: val_macro_f1=0.5738, val_QWK=0.8129
BiLSTM seed 2113, epoch 10: val_macro_f1=0.5767, val_QWK=0.8195
BiLSTM seed 2113, epoch 11: val_macro_f1=0.6152, val_QWK=0.8230
BiLSTM seed 2113, epoch 12: val_macro_f1=0.5633, val_QWK=0.8321
BiLSTM seed 2113, epoch 13: val_macro_f1=0.5831, val_QWK=0.7997
BiLSTM seed 2113, epoch 14: val_macro_f1=0.6042, val_QWK=0.8184
BiLSTM seed 2113, epoch 15: val_macro_f1=0.5904, val_QWK=0.8361
BiLSTM seed 2113, epoch 16: val_macro_f1=0.5996, val_QWK=0.8287
Early stopping seed 2113 at epoch 16; best epoch 11.
BiLSTM seed 2114, epoch 1: val_macro_f1=0.3886, val_QWK=0.7044
BiLSTM seed 2114, epoch 2: val_macro_f1=0.5211, val_QWK=0.

,method,seed,best_epoch,accuracy,balanced_accuracy,macro_f1,quadratic_weighted_kappa,weighted_f1
0,CNN-BiLSTM,2112,10,0.769517,0.669834,0.618879,0.820778,0.781206
1,CNN-BiLSTM,2113,11,0.780669,0.639406,0.615163,0.823009,0.786183
2,CNN-BiLSTM,2114,6,0.737918,0.654949,0.585128,0.812305,0.743239


,precision,recall,f1-score,support
0,0.984791,0.959259,0.971857,270.000000
1,0.579710,0.740741,0.650407,54.000000
2,0.822917,0.533784,0.647541,148.000000
3,0.333333,0.615385,0.432432,26.000000
4,0.322581,0.500000,0.392157,40.000000
accuracy,0.769517,0.769517,0.769517,0.769517
macro avg,0.608666,0.669834,0.618879,538.000000
weighted avg,0.818884,0.769517,0.781206,538.000000


,Grade 0,Grade 1,Grade 2,Grade 3,Grade 4
Grade 0,259,6,4,1,0
Grade 1,1,40,5,2,6
Grade 2,3,17,79,19,30
Grade 3,0,2,2,16,6
Grade 4,0,4,6,10,20


CNN-BiLSTM validation results saved to: /Users/quangduyvu1723/Documents/GitHub/ENGG2112-final-project/outputs/aptos_cnn_bilstm/validation_metrics.json


## Optional final test evaluation — locked by default

Only after the team reviews near-duplicate candidates, documents decisions, and freezes the split, set `CONFIRM_REVIEWED_FROZEN_SPLIT = True` in the configuration cell and run this cell once. It evaluates only the predeclared seed-2112 checkpoint. Do not select settings or seeds using test results.


In [4]:
if not CONFIRM_REVIEWED_FROZEN_SPLIT:
    print('Test evaluation skipped: review and freeze the split before enabling it.')
elif TEST_EVALUATION_COMPLETED:
    raise RuntimeError('Test evaluation was already run in this kernel.')
else:
    TEST_EVALUATION_COMPLETED = True
    test_sequences = extract_feature_sequences(test_frame)
    test_data = TensorDataset(torch.from_numpy(test_sequences), torch.from_numpy(test_frame.diagnosis.to_numpy(dtype=np.int64)))
    test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    primary_model = SpatialBiLSTM().to(RNN_DEVICE)
    saved = torch.load(primary_checkpoint_path, map_location=RNN_DEVICE, weights_only=True)
    if saved.get('manifest_sha256') != MANIFEST_SHA256:
        raise ValueError('Checkpoint does not match the current split manifest.')
    primary_model.load_state_dict(saved['model_state_dict'])
    test_true, test_prediction = predict_sequence_model(primary_model, test_loader)
    test_metrics = calculate_metrics(test_true, test_prediction)
    test_payload = {
        'model': 'CNN-BiLSTM on spatial frozen ResNet-18 features',
        'split_manifest_sha256': MANIFEST_SHA256,
        'confirmation': 'reviewed and frozen by user before evaluation',
        'seed': REPEAT_SEEDS[0],
        'test': test_metrics,
    }
    test_path = OUTPUT_DIR / 'test_metrics_once.json'
    test_path.write_text(json.dumps(test_payload, indent=2), encoding='utf-8')
    print('Final test metrics saved to:', test_path)
    print({key: value for key, value in test_metrics.items() if key != 'per_class'})
    display(pd.DataFrame(test_metrics['per_class']).T)


Test evaluation skipped: review and freeze the split before enabling it.


## Summary and limitations

This notebook trains only the CNN-BiLSTM classifier. It uses spatial features from the shared, frozen ResNet-18 encoder and the existing split. The BiLSTM reads a fixed row-major ordering of image-grid cells; this is one design choice, not a natural temporal order. Report validation seed means and standard deviations, plus per-grade measures. Three runs on one split cannot establish broad generalization. Similar-image review remains incomplete and patient-level independence is unverified. The test set remains locked until review and split freeze.
